In [2]:
import json
import numpy as np
import pandas as pd
from numpy.ma.core import arccos

file_path='./data/data(1).json'
with open(file_path,'r') as f:
    data=json.load(f)

#定义坐标系类
class CoordinateSystem:
    def __init__(self, vector, axis, tasks):
        self.tasks=tasks
        self.vector=np.array(vector,dtype=np.float64)
        self.axis=np.array(axis,dtype=np.float64)
        self.dimensions=vector.shape[1]
        self.num=vector.shape[0]
        if self.vector.shape[1] != self.axis.shape[0]:
            raise ValueError("您输入的向量与坐标系维度不一致")
        return

    def _is_legal_axis(self,axis):
        axis=np.array(axis,dtype=np.float64)
        det=np.linalg.det(self.axis)
        if np.isclose(det,0):
            print('请输入正确的维度，改维度线性相关')
            return False
        else:
            return True

    def change_axis(self, new_axis):
        new_axis=np.array(new_axis,dtype=np.float64)
        if new_axis.shape != (self.dimensions,self.dimensions):
            raise ValueError('维度不匹配')
        if self._is_legal_axis(new_axis):
            neg_axis=np.linalg.inv(self.axis)
            self.vector=np.dot(self.vector,neg_axis)
            self.axis=new_axis
        else:
            raise ValueError('无法构成坐标系')

    def axis_projection(self):
        projection_result=np.zeros((self.dimensions,self.num),dtype=np.float64)
        for i in range(self.dimensions):
            one_axis=self.axis[i]
            for j in range(self.num):
                dot_axis_vector=np.dot(self.vector[j],one_axis)
                dot_axis_axis=np.dot(one_axis,one_axis)
                projection_result[i][j]=dot_axis_vector/dot_axis_axis
        return projection_result



    def axis_angle(self):
        projection_result=self.axis_projection()
        angle_result=np.zeros((self.dimensions,self.num),dtype=np.float64)
        for i in range(self.dimensions):
            one_axis=self.axis[i]
            one_axis_len=np.dot(one_axis,one_axis)
            one_axis_len_sqrt=np.sqrt(one_axis_len)
            for j in range(self.num):
                proj_length = projection_result[i][j] * np.linalg.norm(one_axis)
                vec_len=np.dot(self.vector[j],self.vector[j])
                vec_len_sqrt=np.sqrt(vec_len)
                angle_result[i][j]=arccos(proj_length/(vec_len_sqrt*one_axis_len_sqrt))
        return angle_result

    def area(self):
        det = np.linalg.det(self.axis)
        return abs(det)

    def task(self):
        for i in self.tasks:
            j=i.get('type')
            if j=='change_axis':
                obj_axis=i.get('obj_axis')
                self.change_axis(obj_axis)
            elif j=='axis_angle':
                axis_angle=self.axis_angle()
                print('axis_angle')
            elif j=='axis_projection':
                projection_result=self.axis_projection()
                print('axis_projection')
            else:
                area=self.area()
                print('area')

